# iSWAP vs pump power, 4 Oct 2026 — incoherent error from the clean repetitions

Three sweeps of the resonant SNAIL iSWAP (Q2 donor, Q1 acceptor, SNAIL at 0.715 mA) over 42 pump powers from
−10 to 20 dBm, taken on 4 Oct 2026. The power list is dense where the error has its minimum: −10, −6, −3, 0, 2, 4,
5, 6 dBm, then 6.25 – 12.5 dBm in 0.25 dB steps, then 13 – 20 dBm in 1 dB steps.

| sweep | time (4 Oct) | repetitions | pump frequency and swap time per power |
|---|---|---|---|
| 1 | 02:16 – 04:45 | 4 | interpolated from the 3 Oct mW-linear chevrons |
| 2 | 10:29 – 15:26 | 8 | the 4 hybrid chevrons of 09:33 – 10:27 (fitted below) |
| 3 | 21:09 – 23:37 | 4 (stopped during the 5th) | the 4 hybrid chevrons of 20:14 – 21:08 |

Every repetition: 100 shots per point, **qubit reset 2 ms** (the pumped T1 depends on the shot period — the pump heats a
slow bath, so the duty cycle is held low), 240 pump durations per power on a two-segment grid (dense through the swap
oscillation, ≥ 8 samples per swap period, then sparse out to ~6 T1). Before each repetition a multiplexed readout
calibration and a `T1 and T2` reference at the normal 250 µs reset. Chevrons were taken at the normal reset.

**Fit.** `iswap_incoherent.fit_exchange_pair` on each trace — shared pair 1/T1, 1/Tφ, swap rate, free readout errors
per qubit — with the detuning **fixed at δ = 0** (`delta=0`): the chevrons put the pump on resonance to a few kHz, and a
free δ trades against Tφ (it scattered 0 – 400 kHz between repetitions with χ² unchanged). Incoherent gate error
$1-F \simeq \tfrac45\,t_{\rm iSWAP}\,(1/T_1+1/T_\varphi)$ — only T1 and Tφ during the swap, nothing coherent.

**Clean repetitions.** Some repetitions show an extra dephasing that acts only for pump powers ≈ 7 – 11 dBm and switches
on and off on an hour scale (section 5). Rule fixed before choosing: a repetition is *clean* if its pair Tφ averaged over
7.5 – 10.5 dBm exceeds 50 µs. That gives 10 of 16: sweep 1 all, sweep 2 reps 0, 2, 3, 6, sweep 3 reps 0, 3.

**Which sweeps enter the final numbers.** Sweeps 1 and 2 only (8 clean repetitions). Sweep 3 is kept in the screening,
selection and degradation sections, but even its clean repetitions carry a T1 dip at 11.5 – 12.25 dBm (an error spike
near 12 dBm) that sweeps 1 and 2 do not show, so it is left out of sections 6 – 7.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os, sys, warnings
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor

sys.path.insert(0, "../bell_state")
from iswap_incoherent import US, NS, fit_chevron, fit_traces, screen_traces
from quantify_core.data.handling import set_datadir, load_dataset
DATA_DIR = os.environ.get("QBLOX_DATADIR", "../CD1")  # quantify data dir; override with QBLOX_DATADIR
set_datadir(DATA_DIR)
HERE = os.path.abspath(".")

COLORS = ["#2a78d6", "#1baf7a", "#eda100", "#eb6834", "#7a5cc2"]
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlesize": 10,
    "legend.frameon": False, "legend.fontsize": 8, "lines.linewidth": 1.6,
})

## 1. Data

In [ ]:
SWEEPS = {
    "sweep 1": ["20261004-021656", "20261004-025419", "20261004-033142", "20261004-040904"],
    "sweep 2": ["20261004-102914", "20261004-110636", "20261004-114358", "20261004-122120",
                "20261004-125841", "20261004-133604", "20261004-141325", "20261004-145046"],
    "sweep 3": ["20261004-210936", "20261004-214659", "20261004-222421", "20261004-230143"],
}
T12_REFS = {   # 'T1 and T2' reference taken just before each repetition (250 us reset)
    "sweep 1": ["20261004-021535", "20261004-025258", "20261004-033020", "20261004-040743"],
    "sweep 2": ["20261004-102754", "20261004-110515", "20261004-114238", "20261004-121959",
                "20261004-125720", "20261004-133443", "20261004-141204", "20261004-144926"],
    "sweep 3": ["20261004-210815", "20261004-214537", "20261004-222300", "20261004-230022"],
}
CHEVRONS = ["20261004-093343", "20261004-094714", "20261004-100038", "20261004-101403"]   # before sweep 2

def load_sweep(tuids):
    # donor (Q2, y1) and acceptor (Q1, y0) arrays of shape (repetition, power, duration)
    donor, acc = [], []
    for tu in tuids:
        ds = load_dataset(tu)
        p = np.round(ds.x2.values, 2); pw = np.unique(p); n = np.sum(p == pw[0])
        donor.append(ds.y1.values.reshape(pw.size, n)); acc.append(ds.y0.values.reshape(pw.size, n))
        t = ds.x0.values.reshape(pw.size, n); fp = ds.x1.values.reshape(pw.size, n)[:, 0]
    return pw, t, fp, np.array(donor), np.array(acc)

data = {name: load_sweep(tuids) for name, tuids in SWEEPS.items()}
pw = data["sweep 1"][0]
assert all(np.allclose(d[0], pw) for d in data.values())
mW, Vp = 10 ** (pw / 10), 10 ** (pw / 20)
for name, (p, t, fp, dn, ac) in data.items():
    print("%s: %d repetitions x %d powers x %d durations; span %.0f - %.0f us" % (
        name, *dn.shape, t[:, -1].min() / US, t[:, -1].max() / US))

## 2. Chevrons — resonance and swap time vs power

The four chevrons taken before sweep 2, each power slice fitted in 2-D with `fit_chevron` (detuned exchange model,
shared resonance f0, Ω, T1, Tφ and readout over the slice). They set the pump frequency of sweep 2 and seed the swap time
of every trace fit below. Swap rate $g/2\pi = 1/(4\,t_{\rm iSWAP})$.

In [ ]:
CHEV_CACHE = os.path.join(HERE, "chevron_1004_hybrid_f0.npz")
if os.path.exists(CHEV_CACHE):
    chev = dict(np.load(CHEV_CACHE))
else:
    jobs = []
    for tu in CHEVRONS:
        ds = load_dataset(tu); x0, x1, x2 = ds.x0.values, ds.x1.values, np.round(ds.x2.values, 2)
        for p in pw:
            m = x2 == p; f, t = np.unique(x1[m]), np.unique(x0[m])
            # the chevron's duration axis covers 4 t_swap, so t[-1] / 4 is the swap-time guess
            jobs.append((t, f, ds.y1.values[m].reshape(f.size, t.size), ds.y0.values[m].reshape(f.size, t.size),
                         (t[-1] + t[1]) / 4, f[f.size // 2]))
    with warnings.catch_warnings(), ProcessPoolExecutor(8) as pool:
        warnings.simplefilter("ignore")
        res = list(pool.map(fit_chevron, *zip(*jobs)))
    chev = {k: np.array([r[k] for r in res]).reshape(len(CHEVRONS), pw.size) for k in ("f0", "f0_err", "iswap_time", "redchi")}
    np.savez(CHEV_CACHE, **chev)
f0, f0_sd = chev["f0"].mean(0), chev["f0"].std(0, ddof=1)
tsw, tsw_sd = chev["iswap_time"].mean(0), chev["iswap_time"].std(0, ddof=1)
g_chev = 1 / (4 * tsw)

sel = pw <= 18.0
pf = np.polyfit(mW[sel], f0[sel], 2, w=1 / np.maximum(f0_sd[sel], 5e3))
lo = pw < 10
kappa = np.sum(g_chev[lo] * Vp[lo]) / np.sum(Vp[lo] ** 2)
print("resonance  f0 = %.4f MHz + %.1f kHz/mW P %+.3f kHz/mW^2 P^2 (P <= 18 dBm); chevron scatter median %.1f kHz" % (
    pf[2] / 1e6, pf[1] / 1e3, pf[0] / 1e3, np.median(f0_sd[sel]) / 1e3))
print("swap rate  g/2pi = %.1f kHz x V_p below 10 dBm; t_iSWAP scatter median %.2f ns" % (kappa / 1e3, np.median(tsw_sd) / NS))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].errorbar(mW, f0 / 1e6, yerr=f0_sd / 1e6, fmt="o", ms=3.5, color=COLORS[0], capsize=1.5, label="mean +- sd of 4 chevrons")
pl = np.linspace(0, 100, 200)
ax[0].plot(pl, np.polyval(pf, pl) / 1e6, color=COLORS[3], lw=1, label="quadratic in P, fitted <= 18 dBm")
ax[0].set(xlabel="pump power P (mW at generator)", ylabel="f0 (MHz)", title="exchange resonance"); ax[0].legend()
ax[1].plot(Vp, g_chev / 1e3, "o", ms=3.5, color=COLORS[0], label="1/(4 t_iSWAP)")
ax[1].plot(Vp, kappa * Vp / 1e3, color=COLORS[3], lw=1, label="%.1f kHz x V_p" % (kappa / 1e3))
ax[1].set(xlabel="pump amplitude V_p (sqrt mW)", ylabel="g/2pi (kHz)", title="swap rate"); ax[1].legend()
fig.tight_layout()

## 3. Screening and trace fits

`screen_traces` (preparation and swap correlation with the median of the sweep) plus two checks added after the
readout-phase jumps seen on 3 – 4 Oct: the acceptor (Q1, prepared in |0⟩) must read < 0.1 at zero pump duration, and
the donor + acceptor tail (last 20 points, fully decayed) must be < 0.15. A trace that fails any check is left out of
everything below. Each surviving trace is fitted with `fit_traces(..., delta=0)`, the swap time seeded with the chevron
value.

In [ ]:
FIT_CACHE = os.path.join(HERE, "power_sweep_1004_trace_fits.npz")
cache = dict(np.load(FIT_CACHE)) if os.path.exists(FIT_CACHE) else {}
fits, use = {}, {}
for name, (p, t, fp, dn, ac) in data.items():
    ok, scr = screen_traces(t, dn, ac)
    floor_bad = ac[:, :, 0] > 0.1
    tail_bad = (dn + ac)[:, :, -20:].mean(2) > 0.15
    use[name] = ok & ~floor_bad & ~tail_bad
    key = name.replace(" ", "")
    if key + "_gamma_1" in cache:
        fits[name] = {k[len(key) + 1:]: v for k, v in cache.items() if k.startswith(key + "_")}
    else:
        fits[name] = fit_traces(t, dn, ac, use[name], tsw, delta=0.0)
        cache.update({key + "_" + k: v for k, v in fits[name].items()})
    print("%s: screen %d, floor %d, tail %d failures -> %d of %d traces used" % (
        name, (~ok).sum(), floor_bad.sum(), tail_bad.sum(), use[name].sum(), use[name].size))
np.savez(FIT_CACHE, **cache)

## 4. Clean repetitions

Pair Tφ averaged over 7.5 – 10.5 dBm per repetition, next to the idle coherence of the reference taken just before it.
The degraded repetitions are not degraded at idle — their references are normal — which is why the rule looks at the
pumped Tφ.

In [ ]:
from bell_analysis import coherence_times
win_mid = (pw >= 7.5) & (pw <= 10.5)
CLEAN, rows = {}, []
for name in SWEEPS:
    f, u = fits[name], use[name]
    tphi = np.where(u, 1 / f["gamma_phi"], np.nan); err = np.where(u, f["gate_error"], np.nan)
    CLEAN[name] = []
    for r, tu in enumerate(SWEEPS[name]):
        tp = np.nanmean(tphi[r, win_mid]); clean = tp > 50 * US
        if clean:
            CLEAN[name].append(r)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ref = coherence_times(T12_REFS[name][r])
        rows.append((name, r, tu[9:15], tp / US, np.isfinite(tphi[r, win_mid]).sum(), 100 * np.nanmean(err[r, win_mid]),
                     ref["Q1"]["T1"][0], ref["Q1"]["T2echo"][0], ref["Q2"]["T1"][0], ref["Q2"]["T2echo"][0], clean))
print("%-8s %3s %7s | %-17s %6s | %-24s |" % ("", "rep", "start", "Tphi 7.5-10.5 dBm", "1-F", "idle: Q1 T1 T2E  Q2 T1 T2E"))
for r in rows:
    print("%-8s %3d %7s | %7.1f us (%2d pts) %5.2f%% | %10.1f %4.1f %6.1f %4.1f | %s" % (
        r[0], r[1], r[2], r[3], r[4], r[5], r[6], r[7], r[8], r[9], "clean" if r[10] else "degraded"))
print("\nclean repetitions:", CLEAN, "- total %d of %d" % (sum(map(len, CLEAN.values())), sum(map(len, SWEEPS.values()))))
USE_SWEEPS = ["sweep 1", "sweep 2"]   # sweeps entering sections 6 - 7 (see the introduction)
print("used for the final numbers: %s, %d clean repetitions" % (USE_SWEEPS, sum(len(CLEAN[n]) for n in USE_SWEEPS)))

## 5. Where the degradation acts

In [ ]:
def stack(key, which, sweeps=tuple(SWEEPS)):
    out = []
    for name in sweeps:
        reps = CLEAN[name] if which == "clean" else [r for r in range(len(SWEEPS[name])) if r not in CLEAN[name]]
        if reps:
            out.append(np.where(use[name][reps], fits[name][key][reps], np.nan))
    return np.concatenate(out)

ratio_phi = np.nanmean(stack("gamma_phi", "clean"), 0) / np.nanmean(stack("gamma_phi", "degraded"), 0)   # Tphi degraded / clean
ratio_t1 = np.nanmean(stack("gamma_1", "clean"), 0) / np.nanmean(stack("gamma_1", "degraded"), 0)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(pw, ratio_phi, "s-", ms=3.5, color=COLORS[4], label="Tphi, degraded / clean")
ax.plot(pw, ratio_t1, "o-", ms=3.5, color=COLORS[1], label="T1, degraded / clean")
ax.axhline(1, color="0.4", lw=0.8)
ax.set(xlabel="pump power (dBm)", ylabel="ratio of mean times", ylim=(0.2, 1.4), title="degraded vs clean repetitions (all sweeps)")
ax.legend(); fig.tight_layout()
print("Tphi below 0.7 of the clean value at: %s dBm" % " ".join("%.2f" % p for p in pw[ratio_phi < 0.7]))

## 6. Pair T1, T2, Tφ from the clean repetitions of sweeps 1 and 2

$T_2 = 1/(1/2T_1 + 1/T_\varphi)$. Mean ± standard error over the 8 clean repetitions of sweeps 1 and 2.

In [ ]:
g1, gphi, tsws = (stack(k, "clean", USE_SWEEPS) for k in ("gamma_1", "gamma_phi", "iswap_time"))
T1c, TPc, T2c = 1 / g1, 1 / gphi, 1 / (0.5 * g1 + gphi)
Ec, Erc, Epc = 0.8 * tsws * (g1 + gphi), 0.8 * tsws * g1, 0.8 * tsws * gphi
R = T1c.shape[0]

def ms(X):
    n = np.isfinite(X).sum(0)
    return np.nanmean(X, 0), np.nanstd(X, 0, ddof=1) / np.sqrt(np.maximum(n, 1))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
for X, lab, c, mk in ((T1c, "T1", COLORS[0], "o"), (T2c, "T2", COLORS[1], "^"), (TPc, "Tphi", COLORS[3], "s")):
    m, s = ms(X)
    for a_, x in ((ax[0], pw), (ax[1], mW)):
        a_.errorbar(x, m / US, yerr=s / US, fmt=mk + "-", ms=3.5, lw=0.9, capsize=1.5, color=c, label=lab)
ax[0].set(yscale="log", xlabel="pump power (dBm)", ylabel="time (us)", title="pair T1, T2, Tphi, %d clean repetitions (sweeps 1, 2)" % R)
ax[1].set(yscale="log", xlabel="pump power P (mW at generator)", ylabel="time (us)", title="same vs P")
ax[0].legend(); fig.tight_layout()

### Rates vs pump power

Quadratic in P (mW): a term linear in P is what a three-wave-mixing conversion or photon-number dephasing gives
(∝ pump amplitude²), P² the next order. 1/T1 up to 18 dBm (above it T1 turns back up); 1/Tφ only up to 15 dBm — above
that it rises faster than any low-order polynomial (the swap frequency chirps during the pulse).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for X, lab, c, pmax in ((g1, "1/T1", COLORS[0], 18.0), (gphi, "1/Tphi", COLORS[3], 15.0)):
    m, s = ms(X); sel = pw <= pmax
    po = np.polyfit(mW[sel], m[sel], 2, w=1 / np.maximum(s[sel], 0.03 * m[sel]))
    ax.errorbar(mW[sel], m[sel] / 1e3, yerr=s[sel] / 1e3, fmt="o", ms=3.5, capsize=1.5, color=c)
    pl = np.linspace(0, mW[sel].max(), 200)
    ax.plot(pl, np.polyval(po, pl) / 1e3, color=c, lw=1,
            label="%s = %.1f + %.3f P + %.4f P^2  (1/ms, P in mW, <= %g dBm)" % (lab, po[2] / 1e3, po[1] / 1e3, po[0] / 1e3, pmax))
ax.set(xlabel="pump power P (mW at generator)", ylabel="rate (1/ms)", title="decay rates during the swap, clean repetitions of sweeps 1, 2")
ax.legend(fontsize=7); fig.tight_layout()

## 7. Incoherent error and the optimum (sweeps 1 and 2)

Parabola in dBm through the mean 1 − F within ±1.5 dB of its lowest point, weighted by the standard error; the
uncertainty of the vertex from 2000 bootstrap resamples of the 8 clean repetitions. Sweep 3's clean mean is drawn dashed
for comparison only.

In [ ]:
m, s = ms(Ec)
k0 = int(np.nanargmin(m)); win = np.abs(pw - pw[k0]) <= 1.5
w = 1 / np.maximum(s, 0.03 * m)

def vertex(y):
    c = np.polyfit(pw[win], y[win], 2, w=w[win])
    return -c[1] / (2 * c[0]), np.polyval(c, -c[1] / (2 * c[0])), c

p_opt, e_opt, cpar = vertex(m)
rng = np.random.default_rng(0)
boot = np.array([vertex(np.nanmean(Ec[rng.integers(0, R, R)], 0))[:2] for _ in range(2000)])
boot = boot[np.abs(boot[:, 0] - p_opt) < 5]
print("optimum %.2f +- %.2f dBm, 1-F = %.2f +- %.2f %%  (parabola over %.2f - %.2f dBm, %d clean repetitions)" % (
    p_opt, boot[:, 0].std(), 100 * e_opt, 100 * boot[:, 1].std(), pw[win].min(), pw[win].max(), R))

sel = (pw >= 4) & (pw <= 13)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
for name, c in zip(SWEEPS, (COLORS[2], COLORS[0], COLORS[4])):
    reps = CLEAN[name]
    e = np.nanmean(np.where(use[name][reps], fits[name]["gate_error"][reps], np.nan), 0)
    used = name in USE_SWEEPS
    ax[0].plot(pw[sel], 100 * e[sel], "-" if used else "--", lw=0.9, color=c, alpha=0.6,
               label="%s, clean reps %s%s" % (name, reps, "" if used else " (not used)"))
ax[0].errorbar(pw[sel], 100 * m[sel], yerr=100 * s[sel], fmt="ko-", ms=3.5, lw=1.2, capsize=1.5,
               label="%d clean repetitions of sweeps 1, 2" % R)
pl = np.linspace(pw[win].min(), pw[win].max(), 100)
ax[0].plot(pl, 100 * np.polyval(cpar, pl), color=COLORS[3], lw=1.8,
           label="optimum %.2f +- %.2f dBm, %.2f +- %.2f %%" % (p_opt, boot[:, 0].std(), 100 * e_opt, 100 * boot[:, 1].std()))
ax[0].axvline(9.0, color="0.4", lw=0.8, ls=":")
ax[0].set(xlabel="pump power (dBm)", ylabel="1 - F (%)", title="incoherent iSWAP error 4/5 t_iSWAP (1/T1 + 1/Tphi)")
ax[0].legend(fontsize=7)
ax[1].plot(pw[sel], 100 * np.nanmean(Erc, 0)[sel], "o-", ms=3.5, color=COLORS[1], label="relaxation part")
ax[1].plot(pw[sel], 100 * np.nanmean(Epc, 0)[sel], "s-", ms=3.5, color=COLORS[4], label="dephasing part")
ax[1].plot(pw[sel], 100 * m[sel], "k-", lw=1, label="total")
ax2 = ax[1].twinx()
ax2.plot(pw[sel], np.nanmean(tsws, 0)[sel] / NS, "--", color="0.6", lw=1)
ax2.set_ylabel("t_iSWAP (ns, dashed)", color="0.5"); ax2.grid(False)
ax[1].set(xlabel="pump power (dBm)", ylabel="1 - F (%)", title="error budget"); ax[1].legend()
fig.tight_layout()

print("\n%6s %3s | %7s %7s %7s | %8s | %-13s | %s" % ("dBm", "n", "T1 us", "T2 us", "Tphi", "t_iSWAP", "1-F %", "relax / deph %"))
for p in (4, 5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10, 10.5, 11, 12, 13):
    k = int(np.argmin(abs(pw - p)))
    print("%6.2f %3d | %7.1f %7.1f %7.1f | %6.1f ns | %5.2f +- %4.2f | %.2f / %.2f" % (
        pw[k], np.isfinite(Ec[:, k]).sum(), np.nanmean(T1c, 0)[k] / US, np.nanmean(T2c, 0)[k] / US, np.nanmean(TPc, 0)[k] / US,
        np.nanmean(tsws, 0)[k] / NS, 100 * m[k], 100 * s[k], 100 * np.nanmean(Erc, 0)[k], 100 * np.nanmean(Epc, 0)[k]))

## 8. The T1 dip at 7 dBm

Pair T1 at 7.00 dBm against the mean of its neighbours at 6.75 and 7.25 dBm, per clean repetition. In sweeps 1 and 2 it
sat at 7.00 dBm in every repetition; in the single-qubit T1 with the pump detuned ±5 – 10 MHz it was absent, and in a
resonant scan with the pump stepped ±150 kHz around resonance on the evening of 4 Oct (17:53 – 19:00) it was gone — a
feature tied to the resonant pump that comes and goes.

In [ ]:
kk = [int(np.argmin(abs(pw - p))) for p in (6.75, 7.0, 7.25)]
for name in SWEEPS:
    for r in CLEAN[name]:
        t1 = np.where(use[name][r], 1 / fits[name]["gamma_1"][r], np.nan)[kk] / US
        print("%s rep %d: T1 %.1f / %.1f / %.1f us at 6.75 / 7.00 / 7.25 dBm -> dip %+.1f us" % (
            name, r, *t1, np.mean(t1[[0, 2]]) - t1[1]))

## 9. Summary

* **Optimum** (incoherent part only, 8 clean repetitions of sweeps 1 and 2): 9.09 ± 0.16 dBm with 1 − F = 1.70 ± 0.03 %
  (section 7), within ≈ 0.2 % from about 8.5 to 10 dBm; the 9 dBm operating point sits on it. At 9 dBm: pair T1 39 µs,
  Tφ 60 µs, t_iSWAP 491 ns; relaxation contributes ≈ 1.0 % and dephasing ≈ 0.65 %.
* **Below the optimum** the long gate makes relaxation dominate (pair T1 ≈ 35 – 40 µs from 6 to 10 dBm, with a step
  near 6 dBm — the pump-frequency-dependent Q1 loss edge seen in the detuned-pump T1 map). **Above** it Tφ falls steeply.
* **Degraded state.** In 6 of 16 repetitions Tφ drops to ≈ 0.4 – 0.6 of the clean value between ≈ 7 and 11 dBm while the
  idle references stay normal: a dephasing channel confined to a band of pump powers that switches on and off on an hour
  scale. When on, the error at 9 dBm rises to ≈ 2.4 – 3 %. More repetitions do not average it out; the clean set does.
* **Swap rate** g/2π ≈ 179 kHz per √mW below 10 dBm, growing faster above (odd series in V_p); the resonance moves
  ≈ +96 kHz/mW. Neither Eq. 55 of the Chao Zhou thesis nor its five- and seven-wave terms reproduce the growth of g; the
  three-body simulation in `SNAIL_315_geff.ipynb` does up to ~20 mW.